# 02 System Prompt、Roles 与 Control

## 大模型为什么不是“自由回答”，而是被上下文编排

如果说上一章解决的是“模型为什么看起来像在执行任务”，这一章要解决的问题就是：**是谁在决定它以什么方式执行。**

很多关于 Prompt Engineering 的讨论喜欢把 system prompt 当成一段更高级的话术，把 message roles 当成聊天接口的固定格式。这种理解太浅了。真正在工程系统里起作用的，并不是某几句提示词本身，而是整段上下文被分层组织之后，对模型输出分布形成的控制效果。

从这个角度看，system prompt 不是装饰信息，roles 也不是 UI 层字段。它们共同构成的是一个控制面：决定当前会话里什么信息优先、什么身份有效、什么行为被鼓励、什么输出更像正确答案。

后面无论是 tool calling、Agent loop 还是 MCP resource 读取，都离不开这一层。因为模型从来不是在真空中做决策，它永远只是在对一段被编排过的上下文做条件生成。

## 先给结论

这章最重要的判断是下面这句：

> 大模型的“行为”并不是从参数里直接流出来的，而是被消息结构、角色层级、历史上下文和格式约束持续塑形出来的。

换句话说，模型当然有能力边界，但一次具体会话里的输出样子，很大程度上取决于调用方如何组织上下文。

这意味着三件事：

- system prompt 决定了会话的工作制度，而不仅仅是语气。
- roles 决定了每段信息在上下文中的解释方式，而不是简单标签。
- Agent 的稳定性，本质上依赖上下文编排质量，而不只是模型本身够不够强。

## 1. System Prompt 不是前言，而是制度

对一个任务型系统来说，system prompt 的作用更接近制度设计，而不是提示开场白。

它通常至少在同时定义四件事：

- 身份：当前模型被要求扮演什么角色
- 目标：当前会话最优先完成什么事情
- 边界：什么可以做，什么不应该做
- 程序：遇到不确定情况时，应该直接回答、提问、还是调用外部能力

这四件事一旦写得清楚，模型面对同一个用户请求时，输出风格就会明显收缩到某个更稳定的区间。反过来，如果 system prompt 写得泛泛而谈，模型就会退回到一个更宽、更松、更容易漂移的生成空间。

很多系统不稳定，并不是因为模型本身不行，而是因为调用方只给了一个“帮我回答问题”的模糊环境，却期待模型表现得像一个严格的任务执行器。这个预期从一开始就不成立。

## 2. Roles 不是消息标签，而是解释框架

同样一段文字，放在不同 role 下面，模型对它的解释方式并不相同。

这并不是因为 API 规范强行规定了一个语义宇宙，而是因为训练数据和指令对齐过程让模型学会了：带有不同角色标记的文本，往往承担不同的上下文职责。

在典型会话里，这些角色大致形成如下分工：

- `system`：定义高优先级行为框架
- `user`：提出当前任务、问题或目标
- `assistant`：承接前序推理和中间结论
- `tool`：提供外部执行结果或外部事实反馈

这意味着一件很重要的事：模型看到的从来不是“很多段文字”，而是“很多段在不同制度位置上的文字”。这些位置关系，会直接影响模型判断哪部分信息更像约束、哪部分更像任务、哪部分更像证据。

把同一轮上下文拆开看，会更容易理解 role 的作用：

- `system`：你是严谨的任务型助手，信息不足时优先调用工具，不要猜测。
- `user`：帮我总结这个城市今天的天气，并判断是否适合夜跑。
- `tool`：`temperature=18C; humidity=82%; rain_probability=70%`

这三段文本的关键差异不在内容本身，而在制度位置：第一段是行为边界，第二段是任务目标，第三段是外部结果。

这个例子里最重要的不是三段文本分别写了什么，而是它们被安排在什么位置。

- `system` 不是在提供事实，而是在规定行为准则
- `user` 不是在给规则，而是在定义任务目标
- `tool` 不是在发表意见，而是在提供被系统视为外部结果的输入

同样是“今天可能下雨”，如果这句话出现在 `user` 里，它更像一个待验证的陈述；如果出现在 `tool` 里，它更像一个可据此继续推理的事实反馈。模型之所以会对这些信息采取不同态度，本质上就是在用 role 结构解释上下文。

## 3. 为什么说这是一个 Control Plane

“控制面”这个说法比“提示词技巧”更准确，因为这里控制的不是某句回答，而是整个会话的运行方式。

一个任务系统通常至少要控制以下几个维度：

- 什么时候可以直接回答
- 什么时候必须承认不确定
- 什么时候应该调用工具
- 工具结果回来之后是否允许继续推断
- 最终回答应当更像摘要、解释、决策建议还是结构化输出

这些都不是模型参数层面单独决定的，而是在上下文层面被持续塑形的。system prompt 负责定规则，roles 负责分配语义位置，历史消息负责提供行为惯性，格式约束负责限定最终出口。四者合起来，才构成一个真正可控的 control plane。

## 4. 优先级与冲突，不是附属问题，而是主问题

只要系统不是真空运行，就一定会出现信息冲突。用户可能要求“直接告诉我答案”，而 system prompt 可能要求“缺少证据时必须先调用工具”；历史 assistant 消息可能在前一轮走错了方向，而当前 tool 结果又在纠正它。模型之所以经常表现出不稳定，并不只是因为它会幻觉，更因为它必须在冲突上下文里继续生成。

所以，一个像样的 Agent 系统必须在 prompt 层面提前表达优先级，而不能把优先级问题留给“模型自己悟”。

常见的优先级顺序通常类似这样：

- system 的边界高于 user 的风格诉求
- 外部工具结果高于未经验证的用户假设
- 当前轮明确指令高于过期的历史上下文

这并不是一个法律化、绝对精确的执行语义，但如果调用方不主动提供这种排序，模型就只能在统计意义上自己平衡，结果往往就是时灵时不灵。

把优先级冲突写成一句最小案例就足够了：

- `system` 说：如果事实不确定，必须先使用工具验证，不得直接猜测。
- `user` 说：别查了，直接告诉我结果。
- 正确行为应该是：优先遵循 `system` 的边界，拒绝无依据直答，先说明需要验证或发起工具路径。

这里的重点不是规则写法，而是让读者直接看到：role 之间一旦冲突，系统必须有显式优先级。

## 5. Agent 的稳定性，本质上取决于上下文编排

很多人会把 Agent 的成败归结为“模型够不够强”。这当然有关系，但在实际系统里，更常见的差异来自上下文工程。

同一个模型，在下面两种环境里会呈现出完全不同的稳定性：

- 一个环境只给出模糊任务，没有行为边界，没有工具使用原则，没有结果格式要求
- 另一个环境明确规定身份、约束、失败策略、工具使用时机和回答出口

前者往往像一个偶尔能完成任务的聊天模型，后者更可能像一个行为相对稳定的执行单元。这里的关键变量不是“模型突然更懂了”，而是上下文让模型更容易落到某个可预期的输出分布上。

因此，Agent 设计里一个经常被低估的事实是：**很多所谓“智能体能力”，其实是上下文编排能力。**

## 6. 控制信号会衰减，长上下文不是免费午餐

system prompt 和 role 结构并不意味着控制一定稳定。上下文一旦变长，控制信号就会面临衰减问题。

衰减通常来自几个方向：

- 后续消息数量变多，早期约束被稀释
- 中间步骤引入大量噪音，模型更难分辨真正重要的控制信息
- 工具返回内容过长，事实层信息淹没了制度层信息
- 多轮 assistant 历史形成行为惯性，偏离最初系统边界

这就是为什么成熟的 runtime 很少只把一段 system prompt 永远塞在最开头，然后期待一切正常。很多系统会做控制信息重申、上下文裁剪、摘要回填、工具结果压缩，原因都一样：要保证真正重要的控制信号在后续轮次里仍然可见。

## 7. Prompt 设计的核心不是文案，而是接口治理

把 Prompt Engineering 理解成“如何把一句话说得更聪明”，通常会把问题带偏。对任务系统而言，prompt 的核心价值更接近接口治理：

- 它定义输入在上下文中的解释规则
- 它规定输出应当落入哪些结构区间
- 它帮助 runtime 判断模型回答是否偏离预期

也正因为如此，system prompt 和 message role 其实不是和 tool calling、MCP 分开的主题。恰恰相反，它们是后者成立的前提。因为如果模型连“这类信息是约束、那类信息是工具结果”都解释不稳，后续的一切动作编排都会建立在松软地基上。

## 8. 从 Control Plane 走向 Tool Calling

到这里，其实已经能看出一个关键过渡：模型并不是看到工具就会自动调用，模型首先要在控制面约束下形成一个判断，即“当前问题是否允许直接回答，还是应该走外部能力路径”。

也就是说，tool calling 的前提不是工具存在，而是上下文已经把工具调用塑造成一种更优的输出选择。如果这一层没立住，工具 schema 再漂亮，模型也可能直接乱答、乱选、乱传参数。

所以下一章要讨论的就不是“如何声明一个函数”，而是：**模型为什么会在某一轮选择输出工具调用意图，而不是继续自然语言回答。**

## 9. 本章结论

这一章可以收束成五个判断：

- system prompt 的本质是制度，而不是开场白。
- roles 的本质是上下文解释框架，而不是消息标签。
- 一个任务型会话是否稳定，很大程度上取决于 control plane 是否清晰。
- 优先级、冲突和控制衰减是系统问题，不是边角问题。
- Agent 的很多稳定性表现，本质上来自上下文编排，而不只是模型能力本身。

下一章进入 tool calling。重点不在语法，而在机制：为什么模型会把某次输出变成一个结构化动作意图，以及这个意图如何被外部运行时接住。